In [4]:
import subprocess
subprocess.run(["pip", "install", "pypdf", "langchain-text-splitters", "sentence-transformers", "faiss-cpu", "groq", "python-dotenv"])


CompletedProcess(args=['pip', 'install', 'pypdf', 'langchain-text-splitters', 'sentence-transformers', 'faiss-cpu', 'groq', 'python-dotenv'], returncode=0)

In [5]:
import sys
sys.path.append('../src')
from ingest import load_pdf, chunk_text, create_embeddings
import pickle

text = load_pdf('../data/document.pdf')
print(f'Total characters: {len(text)}')
print('\nFirst 500 chars:\n')
print(text[:500])


c:\Users\arpit\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total characters: 68253

First 500 chars:

User Agreement 
1. Introduction 
This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms 
posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms 
on which eBay offers you access to and use of our Services. You can find an overview of our policies 
here. The Mobile Application Terms of Use, all policies, and additional terms posted on and in our 
Services are incorporated into this User Agreement. You agree


In [6]:
chunks = chunk_text(text)
print(f'Total chunks: {len(chunks)}')
print(f'Avg chunk length: {sum(len(c) for c in chunks) // len(chunks)} chars')
print('\nSample chunk:\n')
print(chunks[0])

Total chunks: 270
Avg chunk length: 254 chars

Sample chunk:

User Agreement 
1. Introduction 
This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms 
posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms


In [7]:
embeddings, model = create_embeddings(chunks)
print(f'Embedding shape: {embeddings.shape}')

Batches: 100%|██████████| 9/9 [00:05<00:00,  1.64it/s]

Embedding shape: (270, 384)


In [8]:
import os
os.chdir('../')
from src.retriever import retrieve
from src.generator import generate_answer

queries = [
    'What is eBay return policy?',
    'Can I sell vehicles on eBay?',
    'What happens if I dont pay for an item?',
    'How does eBay Money Back Guarantee work?',
    'What is the arbitration process?'
]

for q in queries:
    print(f'Q: {q}')
    chunks_retrieved = retrieve(q)
    stream = generate_answer(q, chunks_retrieved)
    answer = ''.join([c.choices[0].delta.content or '' for c in stream])
    print(f'A: {answer}')
    print('-'*60)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2024.42it/s]


Q: What is eBay return policy?
A: According to the provided context, the information about the eBay return policy is not fully mentioned. However, it is mentioned that " buyers can get their money back if an item didn't arrive, is faulty or damaged, or doesn't match the listing."
------------------------------------------------------------
Q: Can I sell vehicles on eBay?
A: You can sell vehicles on eBay, but eBay is not a vehicle broker, dealer, or agent and does not maintain an inventory of vehicles for sale.
------------------------------------------------------------
Q: What happens if I dont pay for an item?
A: Your obligation to pay the seller in the amount of payments received will be satisfied by the eBay Payment Entity, which will then handle the payment, so there's no direct action taken against you, but your payment will be made through the system.
------------------------------------------------------------
Q: How does eBay Money Back Guarantee work?
A: Buyers can get their 